# Pretrain DocNLC trên GPU

Notebook này chuẩn bị config YAML tạm và chạy pipeline pretrain của repo qua `pre_train.py`.

Bạn chỉ cần nhập các đường dẫn quan trọng ở cell cấu hình, đặc biệt là `PRETRAIN_MODEL_G`. Nếu muốn train từ đầu hoàn toàn, để `PRETRAIN_MODEL_G = ""` và `DISTILL = False`.

## 1. Kiểm tra môi trường

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import textwrap
import yaml

import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pre_train.py").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "pre_train.py").exists():
            PROJECT_ROOT = parent
            break

assert (PROJECT_ROOT / "pre_train.py").exists(), "Không tìm thấy root repo DocNLC. Hãy mở notebook từ thư mục DocNLC."
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU 0:", torch.cuda.get_device_name(0))

## 2. Nhập cấu hình pretrain

Format `TRAIN_FILELIST`: mỗi dòng gồm 6 ảnh, phân tách bằng `|`:

```text
GT|WithBack|Blur|Noise|Shadow|Watermark
```

Format `VAL_FILELIST`, nếu dùng validation: mỗi dòng gồm 2 ảnh:

```text
degraded_image|gt_image
```

In [ ]:
# Bắt buộc: filelist cho tập pretrain multi-task.
TRAIN_FILELIST = PROJECT_ROOT / "pretraining_data_path.txt"

# Tùy chọn: để "" nếu chưa muốn chạy validation trong pretrain.
# Lưu ý: pre_train.py hiện hard-code đường dẫn save val image vào /home/wrl/DocNLC/...
# nên mặc định notebook không bật validation.
VAL_FILELIST = ""

# Bạn sẽ tự nhập checkpoint tại đây.
# Nếu để "", notebook sẽ cấu hình train from scratch.
PRETRAIN_MODEL_G = ""  # ví dụ: PROJECT_ROOT / "ckpts" / "some_pretrain.pth"

# Output experiment.
EXPERIMENT_NAME = "BarlowNew_notebook"
OUTPUT_ROOT = PROJECT_ROOT / "output"

# GPU và hyper-parameters chính.
GPU_IDS = [0]
BATCH_SIZE = 8
PATCH_SIZE = 256
NITER = 60000
LR_G = 1e-4
PRINT_FREQ = 40
SAVE_CHECKPOINT_EPOCH = 1

# Nếu PRETRAIN_MODEL_G có checkpoint, có thể bật distill để giống config gốc.
# Trong model Barlow hiện tại phần distill loss đang comment, nên flag này chủ yếu khiến code load thêm netG_Pre.
DISTILL = bool(str(PRETRAIN_MODEL_G).strip())
STRICT_LOAD = False
RESUME_STATE = None

## 3. Validate filelist và checkpoint

In [ ]:
def normalize_optional_path(value):
    if value is None:
        return None
    value = str(value).strip()
    return value or None

def preview_filelist(filelist, expected_cols, name, max_lines=3):
    filelist = Path(filelist)
    assert filelist.exists(), f"Không tìm thấy {name}: {filelist}"
    lines = [line.strip() for line in filelist.read_text().splitlines() if line.strip()]
    assert lines, f"{name} đang rỗng: {filelist}"
    bad = []
    for idx, line in enumerate(lines[:100], 1):
        cols = line.split("|")
        if len(cols) != expected_cols:
            bad.append((idx, len(cols), line))
    if bad:
        idx, n_cols, line = bad[0]
        raise ValueError(f"{name} dòng {idx} có {n_cols} cột, cần {expected_cols}: {line}")
    print(f"{name}: {filelist} ({len(lines)} dòng)")
    print("Preview:")
    for line in lines[:max_lines]:
        print("  ", line)
    return filelist

TRAIN_FILELIST = preview_filelist(TRAIN_FILELIST, 6, "TRAIN_FILELIST")

VAL_FILELIST = normalize_optional_path(VAL_FILELIST)
if VAL_FILELIST:
    VAL_FILELIST = preview_filelist(VAL_FILELIST, 2, "VAL_FILELIST")
else:
    print("VAL_FILELIST: bỏ qua validation trong pretrain.")

PRETRAIN_MODEL_G = normalize_optional_path(PRETRAIN_MODEL_G)
if PRETRAIN_MODEL_G:
    PRETRAIN_MODEL_G = Path(PRETRAIN_MODEL_G)
    assert PRETRAIN_MODEL_G.exists(), f"Không tìm thấy PRETRAIN_MODEL_G: {PRETRAIN_MODEL_G}"
    print("Init từ checkpoint:", PRETRAIN_MODEL_G)
else:
    DISTILL = False
    print("PRETRAIN_MODEL_G rỗng: sẽ train from scratch.")

## 4. Sinh YAML tạm cho pretrain

In [ ]:
config = {
    "name": EXPERIMENT_NAME,
    "use_tb_logger": True,
    "model": "Barlow",
    "distortion": "sr",
    "scale": 1,
    "gpu_ids": GPU_IDS,
    "datasets": {
        "train": {
            "name": "UEN",
            "mode": "multi_task",
            "interval_list": [1],
            "random_reverse": False,
            "border_mode": False,
            "cache_keys": None,
            "filelist": str(TRAIN_FILELIST),
            "use_shuffle": True,
            "n_workers": 0,
            "batch_size": BATCH_SIZE,
            "IN_size": PATCH_SIZE,
            "augment": True,
            "color": "RGB",
        },
    },
    "network_G": {
        "which_model_G": "multi",
        "nf": 16,
        "groups": 8,
    },
    "path": {
        "root": str(OUTPUT_ROOT),
        "results_root": str(OUTPUT_ROOT),
        "pretrain": str(PROJECT_ROOT / "pretrain1"),
        "pretrain_model_G": str(PRETRAIN_MODEL_G) if PRETRAIN_MODEL_G else None,
        "strict_load": STRICT_LOAD,
        "resume_state": RESUME_STATE,
    },
    "train": {
        "lr_G": float(LR_G),
        "lr_scheme": "MultiStepLR",
        "beta1": 0.9,
        "beta2": 0.99,
        "niter": int(NITER),
        "ewc": False,
        "distill": bool(DISTILL),
        "ewc_coff": 50.0,
        "distill_coff": 0.2,
        "fix_some_part": None,
        "warmup_iter": -1,
        "ComputeImportance": False,
        "istraining": True,
        "lr_steps": [60, 90],
        "lr_gamma": 0.5,
        "eta_min": 5e-6,
        "pixel_criterion": "binary",
        "pixel_weight": 5000.0,
        "ssim_weight": 1000.0,
        "vgg_weight": 1000.0,
        "val_epoch": 1,
        "manual_seed": 0,
        "loss_form": "sym",
    },
    "logger": {
        "print_freq": int(PRINT_FREQ),
        "save_checkpoint_epoch": int(SAVE_CHECKPOINT_EPOCH),
    },
}

if VAL_FILELIST:
    config["datasets"]["val"] = {
        "name": "UEN",
        "mode": "UEN_test",
        "filelist": str(VAL_FILELIST),
        "batch_size": 1,
        "use_shuffle": False,
        "IN_size": PATCH_SIZE,
    }

CONFIG_PATH = PROJECT_ROOT / "options" / "pretrain_notebook.yml"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))

print("Đã ghi config:", CONFIG_PATH)
print(CONFIG_PATH.read_text())

## 5. Chạy pretrain

Cell này gọi script gốc `pre_train.py`. Checkpoint sẽ được lưu trong:

`OUTPUT_ROOT / experiments / EXPERIMENT_NAME / models`

In [ ]:
cmd = [sys.executable, "pre_train.py", "--opt", str(CONFIG_PATH), "--launcher", "none"]
print("Running:", " ".join(cmd))
process = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="")
finally:
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"pre_train.py kết thúc với mã lỗi {return_code}")

print("Pretrain hoàn tất.")

## 6. Kiểm tra checkpoint đã sinh

In [ ]:
models_dir = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME / "models"
print("Models dir:", models_dir)
if models_dir.exists():
    checkpoints = sorted(models_dir.glob("*_G.pth"), key=lambda p: p.stat().st_mtime)
    print(f"Tìm thấy {len(checkpoints)} checkpoint G")
    for ckpt in checkpoints[-10:]:
        size_mb = ckpt.stat().st_size / (1024 * 1024)
        print(f"{ckpt.name:>20}  {size_mb:8.2f} MB")
else:
    print("Chưa có thư mục models. Hãy kiểm tra log ở cell pretrain.")